In [1]:
# %%
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, recall_score
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR

# Import statistical and utility functions from util.py
from util import (
    seed_everything, 
    count_parameters, 
    plot_all_model_losses, 
    get_metrics_array,
    compute_delong_paired,
    compute_mcnemar
)
from networks import SpectralViT, SpatialViT, AttentionUNet, SwinTransformer, LogisticRegression, MultiLayerPerceptron
from loader import load_ixi_data, BalancedDataset
from validate import SpectralViT_cv, SpatialViT_cv

# Configuration
MNI_DIR = os.path.expanduser('~/SpectralViT/data/IXI_extracted/')
CSV_PATH = os.path.expanduser('~/SpectralViT/data/IXI_extracted/IXI.csv')
VOL_SIZE = 96
EPOCHS = 500
LR = 1e-4
BATCH_SIZE = 8
N_FOLDS = 5
N_BOOTSTRAP = 1000
N_PERM = 1000
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")

seed_everything(0)

# %%
# Load data
X, Y = load_ixi_data(MNI_DIR, CSV_PATH, VOL_SIZE)
X_flat = X.reshape(len(X), -1)

print(f"Loaded {len(X)} samples")
print(f"Volume shape: {X.shape}")
print(f"Labels: {np.bincount(Y.astype(int))}")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
retrain = False
if retrain is True:
    # %%
    # 1. Config for Selection
    FAST_EPOCHS = 20
    FAST_LR = 1e-3

    # 2. Spatial Selection
    PATCH_SIZE = SpatialViT_cv(
        X=X, Y=Y, 
        patch_candidates=[8, 12, 16], 
        device=device,
        vol_size=96,
        epochs=FAST_EPOCHS,
        lr=FAST_LR,
        physical_bs=2,
        effective_bs=8
    )

    # 3. Spectral Selection
    N_COMP = SpectralViT_cv(
        X_flat=X_flat, Y=Y, 
        pca_candidates=[16, 32, 64, 128], 
        device=device, 
        epochs=FAST_EPOCHS, 
        lr=FAST_LR
    )

    # %%
    # Spectral ViT 
    spectral_vit_history = {'spectral_vit_loss': []}
    oof_probs_spec = []
    oof_y_true = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_flat), 1):
        print(f"Starting Spectral ViT Fold {fold}...")
        torch.cuda.empty_cache()
        
        pca = PCA(n_components=N_COMP, whiten=True).fit(X_flat[train_idx])
        tr_pca = torch.from_numpy(pca.transform(X_flat[train_idx])).float()
        tr_y = torch.from_numpy(Y[train_idx]).float()
        ts_pca = torch.from_numpy(pca.transform(X_flat[test_idx])).float().to(device)
        ts_y = torch.from_numpy(Y[test_idx]).float().to(device)
        
        loader = DataLoader(TensorDataset(tr_pca, tr_y), batch_size=BATCH_SIZE, shuffle=True)
        model = SpectralViT(n_inputs=N_COMP, embed_dim=16, use_mode_weights=True).to(device)
        
        opt = optim.AdamW(model.parameters(), lr=LR)
        sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader), epochs=EPOCHS)
        crit = nn.BCEWithLogitsLoss()
        
        spectral_vit_fold_losses = []
        for epoch in range(EPOCHS):
            model.train()
            epoch_loss = 0
            for b_pca, b_y in loader:
                b_pca, b_y = b_pca.to(device), b_y.to(device)
                opt.zero_grad()
                loss = crit(model(b_pca), b_y)
                loss.backward()
                epoch_loss += loss.item()
                opt.step()
                sched.step()
            spectral_vit_fold_losses.append(epoch_loss / len(loader))
            
        spectral_vit_history['spectral_vit_loss'].append(spectral_vit_fold_losses)
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(ts_pca)).cpu().numpy()
            oof_probs_spec.append(probs)
            oof_y_true.append(ts_y.cpu().numpy())

    params_spec = count_parameters(model)
    print(f"Spectral ViT parameters: {params_spec:,}")

    # %%
    # Spatial ViT Configuration
    spatial_vit_history = {'spatial_vit_loss': []}
    oof_probs_spat = []
    PHYSICAL_BATCH_SIZE = 2
    EFFECTIVE_BATCH_SIZE = 8
    ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE

    for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
        print(f"Starting Spatial ViT Fold {fold}...")
        torch.cuda.empty_cache()
        
        tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
        tr_y = torch.from_numpy(Y[train_idx]).float()
        ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
        ts_y = torch.from_numpy(Y[test_idx]).float().to(device)
        
        loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=PHYSICAL_BATCH_SIZE, shuffle=True)
        model = SpatialViT(vol_size=VOL_SIZE, patch_size=PATCH_SIZE, embed_dim=128).to(device)
        
        opt = optim.AdamW(model.parameters(), lr=LR)
        sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader)//ACCUMULATION_STEPS, epochs=EPOCHS)
        crit = nn.BCEWithLogitsLoss()
        
        fold_losses = []
        for epoch in range(EPOCHS):
            model.train()
            epoch_loss = 0
            for i, (b_vol, b_y) in enumerate(loader):
                b_vol, b_y = b_vol.to(device), b_y.to(device)
                logits = model(b_vol)
                loss = crit(logits, b_y) / ACCUMULATION_STEPS
                loss.backward()
                epoch_loss += loss.item() * ACCUMULATION_STEPS
                if (i + 1) % ACCUMULATION_STEPS == 0:
                    opt.step(); sched.step(); opt.zero_grad()
            fold_losses.append(epoch_loss / len(loader))
        
        spatial_vit_history['spatial_vit_loss'].append(fold_losses)
        model.eval()
        with torch.no_grad():
            test_probs = []
            for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                test_probs.append(torch.sigmoid(model(chunk)))
            oof_probs_spat.append(torch.cat(test_probs).cpu().numpy())

    params_spat = count_parameters(model)

    # %%
    # Compact Spatial ViT 
    compact_spatial_vit_history = {'compact_spatial_vit_loss': []}
    oof_probs_spat_matched = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
        print(f"Starting Compact Spatial ViT Fold {fold}...")
        torch.cuda.empty_cache()
        
        tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
        tr_y = torch.from_numpy(Y[train_idx]).float()
        ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
        ts_y = torch.from_numpy(Y[test_idx]).float().to(device)
        
        loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=PHYSICAL_BATCH_SIZE, shuffle=True)
        model = SpatialViT(vol_size=VOL_SIZE, patch_size=12, embed_dim=12, n_heads=1, n_layers=1, is_2d=False, use_cls_token=False).to(device)
        
        opt = optim.AdamW(model.parameters(), lr=LR)
        sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader)//ACCUMULATION_STEPS, epochs=EPOCHS)
        crit = nn.BCEWithLogitsLoss()
        
        fold_losses = []
        for epoch in range(EPOCHS):
            model.train()
            epoch_loss = 0
            for i, (b_vol, b_y) in enumerate(loader):
                b_vol, b_y = b_vol.to(device), b_y.to(device)
                loss = crit(model(b_vol), b_y) / ACCUMULATION_STEPS
                loss.backward()
                epoch_loss += loss.item() * ACCUMULATION_STEPS
                if (i + 1) % ACCUMULATION_STEPS == 0:
                    opt.step(); sched.step(); opt.zero_grad()
            fold_losses.append(epoch_loss / len(loader))
        compact_spatial_vit_history['compact_spatial_vit_loss'].append(fold_losses)
        
        model.eval()
        with torch.no_grad():
            test_probs = []
            for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                test_probs.append(torch.sigmoid(model(chunk)))
            oof_probs_spat_matched.append(torch.cat(test_probs).cpu().numpy())

    params_match = count_parameters(model)

    # %%
    # Swin ViT
    oof_probs_swin = []
    swin_history = {'swin_loss': []}
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        tr_vol, tr_y = torch.from_numpy(X[train_idx]).unsqueeze(1).float(), torch.from_numpy(Y[train_idx]).float()
        ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
        train_loader = DataLoader(BalancedDataset(tr_vol, tr_y), batch_size=BATCH_SIZE, shuffle=True)
        m_swin = SwinTransformer(img_size=X.shape[-1], patch_size=8, window_size=6).to(device)
        optimizer = optim.AdamW(m_swin.parameters(), lr=1e-4)
        criterion = nn.BCEWithLogitsLoss()

        fold_losses = []
        for epoch in range(EPOCHS):
            m_swin.train()
            epoch_loss = 0
            for b_vol, b_y in train_loader:
                b_vol, b_y = b_vol.to(device), b_y.to(device)
                optimizer.zero_grad()
                l = criterion(m_swin(b_vol + torch.randn_like(b_vol)*0.01), b_y)
                l.backward(); optimizer.step()
                epoch_loss += l.item()
            fold_losses.append(epoch_loss / len(train_loader))
        swin_history['swin_loss'].append(fold_losses)
        with torch.no_grad():
            m_swin.eval()
            oof_probs_swin.append(torch.sigmoid(m_swin(ts_vol)).cpu().numpy())
        del m_swin; gc.collect(); torch.cuda.empty_cache()

    params_swin = count_parameters(SwinTransformer(img_size=X.shape[-1], patch_size=8, window_size=6).to(device))

    # %%
    # U-net
    unet_history = {'unet_loss': []}
    oof_probs_unet = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        tr_vol, tr_y = torch.from_numpy(X[train_idx]).unsqueeze(1).float(), torch.from_numpy(Y[train_idx]).float()
        ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
        train_loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=BATCH_SIZE, shuffle=True)
        m_unet = AttentionUNet(in_channels=1, base_channels=7).to(device)
        optimizer, criterion = optim.AdamW(m_unet.parameters(), lr=1e-4), nn.BCEWithLogitsLoss()

        fold_losses = []
        for epoch in range(EPOCHS):
            m_unet.train(); epoch_loss = 0
            for b_vol, b_y in train_loader:
                b_vol, b_y = b_vol.to(device), b_y.to(device)
                optimizer.zero_grad(); loss = criterion(m_unet(b_vol), b_y)
                loss.backward(); optimizer.step(); epoch_loss += loss.item()
            fold_losses.append(epoch_loss / len(train_loader))
        unet_history['unet_loss'].append(fold_losses)
        m_unet.eval()
        with torch.no_grad():
            test_probs = []
            for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                test_probs.append(torch.sigmoid(m_unet(chunk)))
            oof_probs_unet.append(torch.cat(test_probs).cpu().numpy())

    params_unet = count_parameters(m_unet)

    # %%
    plot_all_model_losses(spectral_vit_history, spatial_vit_history, compact_spatial_vit_history, unet_history, swin_history, EPOCHS)

# %%
else:
    # Master Load Cell (Restoration)
    filename = 'spectral_vit_master_state_se.pth'
    if os.path.exists(filename):
        master_state = torch.load(filename, map_location='cpu')
        results = master_state.get('results', {})
        oof_probs_spec = results.get('oof_probs_spec')
        oof_probs_spat = results.get('oof_probs_spat')
        oof_probs_spat_matched = results.get('oof_probs_spat_matched')
        oof_probs_swin = results.get('oof_probs_swin')
        oof_probs_unet = results.get('oof_probs_unet')
        oof_probs_lr = results.get('oof_probs_lr')
        oof_probs_mlp = results.get('oof_probs_mlp')
        oof_y_true = results.get('oof_y_true')
        config = master_state.get('config', {})
        N_COMP = config.get('N_COMP')
        print(f"✅ MASTER STATE LOADED from '{filename}'.")

# Spectral Baselines
seed_everything(0)
MLP_HIDDEN_DIM = 16
oof_probs_lr, oof_probs_mlp = [], []
y_final = np.concatenate(oof_y_true)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_flat), 1):
    pca = PCA(n_components=N_COMP, whiten=True).fit(X_flat[train_idx])
    t_X_tr = torch.from_numpy(pca.transform(X_flat[train_idx])).float().to(device)
    t_y_tr = torch.from_numpy(Y[train_idx]).float().to(device).unsqueeze(1)
    t_X_ts = torch.from_numpy(pca.transform(X_flat[test_idx])).float().to(device)
    
    lr_model = LogisticRegression(N_COMP).to(device)
    lr_opt, lr_crit = optim.AdamW(lr_model.parameters(), lr=LR), nn.BCEWithLogitsLoss()
    for _ in range(500): 
        lr_opt.zero_grad(); lr_crit(lr_model(t_X_tr), t_y_tr).backward(); lr_opt.step()
    with torch.no_grad(): oof_probs_lr.append(torch.sigmoid(lr_model(t_X_ts)).cpu().numpy().flatten())
    
    mlp_model = MultiLayerPerceptron(input_dim=N_COMP, hidden_dim=MLP_HIDDEN_DIM).to(device)
    mlp_opt, mlp_crit = optim.AdamW(mlp_model.parameters(), lr=LR), nn.BCEWithLogitsLoss()
    for _ in range(500): 
        mlp_opt.zero_grad(); mlp_crit(mlp_model(t_X_tr), t_y_tr).backward(); mlp_opt.step()
    with torch.no_grad(): oof_probs_mlp.append(torch.sigmoid(mlp_model(t_X_ts)).cpu().numpy().flatten())

params_lr = N_COMP + 1
params_mlp = (N_COMP * MLP_HIDDEN_DIM + MLP_HIDDEN_DIM) + (MLP_HIDDEN_DIM * 1 + 1)

# %%
# =============================================================================
# FINAL STATISTICAL EVALUATION: BOOTSTRAP CIs & ONE-SIDED PERMUTATION TEST
# =============================================================================

def compute_metrics_comprehensive(y_true, y_prob, threshold=0.5):
    """Internal helper for comprehensive metric calculation."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'B-Acc': (sens + spec) / 2,
        'Spec': spec,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# Gather overall predictions
y_final = np.concatenate(oof_y_true)
all_model_probs = {
    "Spec-ViT": np.concatenate(oof_probs_spec),
    "PCA+LR": np.concatenate(oof_probs_lr),
    "PCA+MLP": np.concatenate(oof_probs_mlp),
    "Spat-H": np.concatenate(oof_probs_spat),
    "Spat-M": np.concatenate(oof_probs_spat_matched),
    "Swin": np.concatenate(oof_probs_swin),
    "U-Net": np.concatenate(oof_probs_unet)
}
model_names = list(all_model_probs.keys())
ref_name = "Spec-ViT"
n_samples = len(y_final)

# 1. Point Estimates
obs_metrics = {name: compute_metrics_comprehensive(y_final, all_model_probs[name]) for name in model_names}

# 2. Bootstrap Confidence Intervals
print(f"\nRunning {N_BOOTSTRAP} Bootstrap iterations for 95% CIs...")
boot_stats = {name: {k: [] for k in ['AUC', 'B-Acc', 'Spec', 'F1']} for name in model_names}
np.random.seed(0)
for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(n_samples, n_samples, replace=True)
    if len(np.unique(y_final[idx])) < 2: continue
    for name in model_names:
        m = compute_metrics_comprehensive(y_final[idx], all_model_probs[name][idx])
        for k in ['AUC', 'B-Acc', 'Spec', 'F1']:
            boot_stats[name][k].append(m[k])

# 3. Paired One-Sided Permutation Test (H1: Spectral > Model)
print(f"Running {N_PERM} Permutations for One-Sided Significance vs {ref_name}...")
p_values = {name: {k: 1.0 for k in ['AUC', 'B-Acc', 'Spec', 'F1']} for name in model_names}

for name in model_names:
    if name == ref_name: continue
    for k in ['AUC', 'B-Acc', 'Spec', 'F1']:
        obs_diff = obs_metrics[ref_name][k] - obs_metrics[name][k]
        
        perm_diffs = np.zeros(N_PERM)
        for i in range(N_PERM):
            swap = np.random.rand(n_samples) < 0.5
            # Paired swap between ref and current model
            p_a = np.where(swap, all_model_probs[name], all_model_probs[ref_name])
            p_b = np.where(swap, all_model_probs[ref_name], all_model_probs[name])
            
            m_a = compute_metrics_comprehensive(y_final, p_a)
            m_b = compute_metrics_comprehensive(y_final, p_b)
            perm_diffs[i] = m_a[k] - m_b[k]
        
        # One-sided: p is the fraction of permutations where the random diff is >= observed lead
        if obs_diff > 0:
            p_values[name][k] = np.mean(perm_diffs >= obs_diff)
        else:
            p_values[name][k] = 1.0

# 4. Print results table
col_width = 24
table_width = 16 + (4 * col_width)
print("\n" + "=" * table_width)
print(f"{'Model':<16} | {'AUC [95% CI]':^{col_width}} | {'B-Acc [95% CI]':^{col_width}} | {'Spec [95% CI]':^{col_width}} | {'F1 [95% CI]':^{col_width}}")
print("-" * table_width)

for name in model_names:
    row = f"{name:<16}"
    for k in ['AUC', 'B-Acc', 'Spec', 'F1']:
        val = obs_metrics[name][k]
        low, high = np.percentile(boot_stats[name][k], [2.5, 97.5])
        sig = "*" if (name != ref_name and p_values[name][k] < 0.05) else ""
        row += f" | {val:.3f} [{low:.2f},{high:.2f}]{sig:<1}"
    print(row)

print("-" * table_width)
# param_row = f"{'Parameters':<16} | "
# param_vals = []
# param_counts = {
#     "Spec-ViT": params_spec, "PCA+LR": params_lr, "PCA+MLP": params_mlp,
#     "Spat-H": params_spat, "Spat-M": params_match, "Swin": params_swin, "U-Net": params_unet
# }
# for name in model_names:
#     param_row += f"{param_counts[name]:^{col_width},}" + " | "
# print(param_row.rstrip(" | "))
print("=" * table_width)
print(f"(*) One-sided paired permutation p < 0.05 comparing {ref_name} vs Model (H1: {ref_name} > Model).")

Loading Data: 100%|██████████| 581/581 [02:06<00:00,  4.58it/s]


Loaded 566 samples
Volume shape: (566, 96, 96, 96)
Labels: [254 312]
✅ MASTER STATE LOADED from 'spectral_vit_master_state_se.pth'.

Running 1000 Bootstrap iterations for 95% CIs...
Running 1000 Permutations for One-Sided Significance vs Spec-ViT...

Model            |       AUC [95% CI]       |      B-Acc [95% CI]      |      Spec [95% CI]       |       F1 [95% CI]       
----------------------------------------------------------------------------------------------------------------
Spec-ViT         | 0.848 [0.81,0.88]  | 0.806 [0.77,0.84]  | 0.760 [0.71,0.81]  | 0.833 [0.80,0.86] 
PCA+LR           | 0.700 [0.66,0.74]* | 0.617 [0.58,0.66]* | 0.567 [0.50,0.63]* | 0.660 [0.62,0.70]*
PCA+MLP          | 0.793 [0.76,0.83]* | 0.733 [0.69,0.77]* | 0.748 [0.69,0.80]  | 0.747 [0.71,0.78]*
Spat-H           | 0.737 [0.70,0.78]* | 0.679 [0.64,0.72]* | 0.614 [0.55,0.68]* | 0.723 [0.68,0.76]*
Spat-M           | 0.654 [0.61,0.70]* | 0.640 [0.60,0.68]* | 0.606 [0.54,0.67]* | 0.675 [0.63,0.72]*
Swin  